# Demo 5: Investor Dashboard

**Product and IP: dashboard and hardware roadmap**

Zero-code investor view over Demos 1–4:
- Polysemanticity collapse (baseline vs. ensemble ∩)
- Spike–hypergraph topology
- Causal verification (STII + ACDC)
- Fairness indication

_This notebook mirrors the Streamlit dashboard for static sharing._

In [ ]:
import sys, os, json, platform
sys.path.append(os.path.abspath("."))  # make ./python importable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("networkx:", nx.__version__)

In [ ]:
from python.utils.config import load_yaml  # configs/demo5_dashboard.yaml
from python.utils.artifacts import create_run_dir, dump_json
from python.dashboard.run_discovery import resolve_selection
from python.dashboard.hif_utils import load_metrics, load_hif, summarize_hif
from python.dashboard.plots import hypergraph_small_graph, stii_bar

cfg = load_yaml("configs/demo5_dashboard.yaml")
sel = resolve_selection(cfg)
print("Selection:\n", json.dumps(sel, indent=2))

# outputs/investor/<timestamp>
out_dir = create_run_dir("outputs/investor")
plots_dir = os.path.join(out_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)
print("Investor outputs:", out_dir)

In [ ]:
metrics_by = {}
for key in ("baseline", "ensemble", "spike_hypergraph", "causal"):
    path = sel.get(key)
    if path:
        metrics_by[key] = load_metrics(path)
    else:
        metrics_by[key] = {}
print("Loaded keys:", {k: list(v.keys()) for k, v in metrics_by.items()})

In [ ]:
from python.plots.hist import plot_histogram
from python.plots.compare import plot_dual_hist
from IPython.display import Image, display

bins = int(cfg.get("plots", {}).get("hist_bins", 30))

# Baseline histogram
base_run = sel.get("baseline")
if base_run:
    poly_counts_path = os.path.join(base_run, "poly_counts.npy")
    if os.path.exists(poly_counts_path):
        arr = np.load(poly_counts_path)
        out_png = os.path.join(plots_dir, "baseline_poly_hist.png")
        plot_histogram(arr, bins, "Baseline poly(f) counts", out_png)
        display(Image(filename=out_png))
    else:
        print("Baseline: poly_counts.npy not found")
else:
    print("Baseline run not selected")

# Ensemble overlay (single vs. intersection)
ens_run = sel.get("ensemble")
if ens_run:
    a_single = os.path.join(ens_run, "poly_counts_single.npy")
    a_inter = os.path.join(ens_run, "poly_counts_intersection.npy")
    if os.path.exists(a_single) and os.path.exists(a_inter):
        arr_a = np.load(a_single)
        arr_b = np.load(a_inter)
        out_png = os.path.join(plots_dir, "ensemble_dual_hist.png")
        plot_dual_hist(arr_a, arr_b, bins, ("Single SAE", "Ensemble ∩"), "Polysemanticity — Single vs. ∩", out_png)
        display(Image(filename=out_png))
    else:
        if os.path.exists(a_single):
            out_png = os.path.join(plots_dir, "ensemble_single_hist.png")
            plot_histogram(np.load(a_single), bins, "Single SAE poly(f)", out_png)
            display(Image(filename=out_png))
        if os.path.exists(a_inter):
            out_png = os.path.join(plots_dir, "ensemble_intersection_hist.png")
            plot_histogram(np.load(a_inter), bins, "Ensemble ∩ poly(f)", out_png)
            display(Image(filename=out_png))
else:
    print("Ensemble run not selected")

# Hypergraph histogram (use existing image if present)
hyp_run = sel.get("spike_hypergraph")
if hyp_run:
    hpng = os.path.join(hyp_run, "poly_hist_hyperedges.png")
    if os.path.exists(hpng):
        display(Image(filename=hpng))
    else:
        print("Hypergraph: histogram image not found")
else:
    print("Hypergraph run not selected")

In [ ]:
# HIF topology (prefer causal HIF with STII, else hypergraph HIF)
from IPython.display import Image, display

hif_path = None
for candidate in ("causal", "spike_hypergraph"):
    rd = sel.get(candidate)
    if rd:
        m = metrics_by.get(candidate, {})
        p = m.get("hif_path")
        if p and os.path.exists(p):
            hif_path = p
            break

if hif_path:
    hif = load_hif(hif_path)
    s = summarize_hif(hif)
    print("HIF summary:\n", json.dumps(s, indent=2))
    fig = hypergraph_small_graph(hif, top_k_edges=25)
    topo_png = os.path.join(plots_dir, "topology_small.png")
    fig.savefig(topo_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=topo_png))
else:
    print("No HIF available in selected runs")

In [ ]:
# STII and ACDC
top_k = int(cfg.get("plots", {}).get("top_k_stii", 20))
c_run = sel.get("causal")
stii_bar_png = None
causal_summary = {}
if c_run and metrics_by.get("causal"):
    stii = metrics_by["causal"].get("causal_stii", {})
    vals = list(stii.get("values", [])) if isinstance(stii, dict) else []
    if vals:
        vals.sort(key=lambda d: (float(d.get("stii", 0.0)), str(d.get("edge", []))), reverse=True)
        top = vals[:top_k]
        items = [("{" + ",".join(map(str, (d.get("edge") or []))) + "}", float(d.get("stii", 0.0))) for d in top]
        fig = stii_bar(items, title=f"Top-{top_k} STII edges")
        stii_bar_png = os.path.join(plots_dir, "stii_topk_bar.png")
        fig.savefig(stii_bar_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        from IPython.display import Image, display
        display(Image(filename=stii_bar_png))
    acdc = metrics_by["causal"].get("acdc", {})
    if acdc:
        causal_summary = {
            "base_acc": float(acdc.get("base_acc", 0.0)),
            "final_acc": float(acdc.get("final_acc", 0.0)),
            "kept_edges": int(len(acdc.get("kept_edges", []))),
        }
        print("ACDC:", causal_summary)
    else:
        print("ACDC minimal circuit not present")
else:
    print("No causal run selected or metrics missing")

In [ ]:
# Fairness report
fair_summary = {}
if c_run and metrics_by.get("causal"):
    fair = metrics_by["causal"].get("fairness", {})
    if fair:
        fair_summary = {
            "num_biased_nodes": int(fair.get("num_biased_nodes", 0)),
            "biased_nodes_in_minimal_count": int(fair.get("biased_nodes_in_minimal_count", 0)),
            "any_biased_node_in_minimal": bool(fair.get("any_biased_node_in_minimal", False)),
        }
        print("Fairness summary:", fair_summary)
        examples = fair.get("examples", [])[: int(cfg.get("plots", {}).get("top_k_gender_nodes", 10))]
        if examples:
            import pandas as pd
            df = pd.DataFrame([
                {
                    "node_id": ex.get("node_id"),
                    "p_male": ex.get("p_male"),
                    "p_female": ex.get("p_female"),
                    "in_minimal": ex.get("in_minimal"),
                }
                for ex in examples
            ])
            display(df)
    else:
        print("No fairness_report.json")
else:
    print("No causal fairness metrics loaded")

In [ ]:
# Save combined summary artifacts
combined = {
    "selection": sel,
    "baseline": metrics_by.get("baseline", {}).get("baseline"),
    "ensemble_single": metrics_by.get("ensemble", {}).get("ensemble_single"),
    "ensemble_intersection": metrics_by.get("ensemble", {}).get("ensemble_intersection"),
    "hypergraph": metrics_by.get("spike_hypergraph", {}).get("hypergraph"),
    "causal_stii": metrics_by.get("causal", {}).get("causal_stii"),
    "acdc": metrics_by.get("causal", {}).get("acdc"),
    "fairness": metrics_by.get("causal", {}).get("fairness"),
    "artifacts": {
        "baseline_poly_hist": os.path.join(plots_dir, "baseline_poly_hist.png") if os.path.exists(os.path.join(plots_dir, "baseline_poly_hist.png")) else None,
        "ensemble_dual_hist": os.path.join(plots_dir, "ensemble_dual_hist.png") if os.path.exists(os.path.join(plots_dir, "ensemble_dual_hist.png")) else None,
        "topology_small": topo_png if 'topo_png' in globals() and os.path.exists(topo_png) else None,
        "stii_topk_bar": stii_bar_png if 'stii_bar_png' in globals() and stii_bar_png and os.path.exists(stii_bar_png) else None,
    }
}
dump_json(combined, os.path.join(out_dir, "dashboard_metrics.json"))

# Markdown summary
lines = []
lines.append("# Investor Dashboard Summary")
lines.append("")
lines.append(f"Output directory: `{out_dir}`")
lines.append("## Acceptance mapping (DEMO_CORRIDOR.md)")
lines.append("- Polysemanticity collapse via baseline+ensemble histograms")
lines.append("- Spike–hypergraph topology with counts and small graph")
lines.append("- Causal circuits with STII top edges and ACDC accuracy")
lines.append("- Fairness indication from minimal circuit")
with open(os.path.join(out_dir, "dashboard_summary.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Wrote dashboard_metrics.json and dashboard_summary.md")

### Acceptance Mapping (DEMO_CORRIDOR.md)
- Polysemanticity collapse: Overlay histograms (single vs. ∩) and summary deltas.
- Hypergraph topology: Node/edge counts and a small bipartite view.
- Causal circuits: Top STII edges and base→final accuracy (ACDC).
- Fairness: Presence of gender-associated nodes in minimal circuit.